# Day 1 · 01 · From SQL Warehouse to Trusted Gold Layer

![Medallion to Power BI](../../assets/images/medallion_to_powerbi.png)

This notebook starts the full RetailHub analytics story: explore data in
Databricks SQL, understand SQL Warehouse cost drivers, explain the Medallion
Architecture, and build the first Gold layer that Power BI can trust.

## Business Scenario

RetailHub wants an executive KPI dashboard in Power BI. The current reporting
process has three problems:

1. different reports calculate revenue differently,
2. report performance changes depending on the visual and filter,
3. nobody can point to one approved source of truth.

The training answer is to build a governed Gold layer in Databricks and make
Power BI a thin reporting layer.

## Learning Objectives

By the end of this notebook you can:

- explain the analyst workflow in Databricks SQL,
- use SQL Editor, Catalog Explorer and Query History as analyst tools,
- identify what drives SQL Warehouse cost and latency,
- compare Serverless, Pro and Classic SQL Warehouses at decision level,
- describe Bronze, Silver and Gold responsibilities,
- explain where Medallion fits next to Kimball, Inmon, Data Vault and Data Mesh,
- discover source grain and common data-quality risks,
- build a first Kimball-style Gold layer,
- explain why Gold is a BI contract, not only a table.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_product",
    f"{GOLD}.dim_customer",
    f"{GOLD}.fact_sales",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: data/generate_training_dataset.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"data/generate_training_dataset.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 1. SQL Warehouse orientation

![SQL Warehouse cost decision](../../assets/images/sql_warehouse_cost_decision.png)

Analysts usually experience Databricks through SQL Editor, notebooks, query
history and dashboards. For this course, the important operational idea is:
the warehouse is the compute layer behind SQL interactions.

| Area | What to show | Why it matters |
|---|---|---|
| SQL Editor | where ad-hoc SQL is written | fast exploration |
| Warehouse settings | size, type, auto-stop | direct cost impact |
| Query History | duration, user, warehouse | performance and cost diagnosis |
| Catalog Explorer | tables, comments, lineage | BI trust and discoverability |

`[UI DEMO]` Open SQL Warehouse settings and Query History before running the
next profiling queries.

## 2. SQL Editor Productivity Lab

![SQL Editor](../../assets/images/source_sql_editor.png)

This is a short live demo, not a separate SQL Editor course. The goal is to
show how a Power BI developer or analyst works day to day before the modelling
starts.

For a Power BI developer, SQL Editor is the fastest way to answer practical
questions before a report is built:

| Question | SQL Editor action | Why it matters for Power BI |
|---|---|---|
| Which table should I connect to? | inspect catalog, schema and table comments | avoids connecting to a staging or Silver table by accident |
| What is the grain? | run count and distinct-count checks | prevents wrong relationships and inflated measures |
| Is a filter selective? | test date/status/channel predicates | predicts Import refresh size and DirectQuery latency |
| Is the query expensive? | open Query History after execution | links report design to warehouse cost |
| Can another developer reuse this logic? | save/query/comment the SQL | keeps BI logic outside individual PBIX files |

`[UI DEMO]` In SQL Editor, show:

1. **Catalog / schema picker** - select the training catalog and `silver` /
   `gold` schemas.
2. **Multi-statement results** - run two `SELECT` statements and compare their
   result tabs.
3. **Inline execution history** - find the last query and open its details.
4. **Databricks Assistant** - ask for a short explanation of a query, but do
   not let it replace validation.
5. **Default warehouse** - confirm which SQL Warehouse this analyst will use.

Trainer note: this 10-15 minute demo is what makes the course feel like a
real Databricks analyst workflow instead of only a notebook exercise.

### Hands-on: parameterized exploration query

The SQL Editor supports parameters. In a notebook we simulate the same idea
with a widget-backed value. Use this to show how analysts narrow a query before
they hand it to a dashboard.

In [ ]:
available_months = [
    r["year_month"]
    for r in spark.sql(f"""
        SELECT DISTINCT date_format(order_date, 'yyyy-MM') AS year_month
        FROM {SILVER}.sales_orders
        WHERE order_date IS NOT NULL
        ORDER BY year_month
        LIMIT 24
    """).collect()
]

default_month = available_months[-1] if available_months else "2025-01"

try:
    dbutils.widgets.dropdown("p_year_month", default_month, available_months or [default_month])
    p_year_month = dbutils.widgets.get("p_year_month")
except Exception:
    p_year_month = default_month

print("Selected month:", p_year_month)

spark.sql(f"""
SELECT
  date_format(o.order_date, 'yyyy-MM') AS year_month,
  o.channel,
  o.status,
  COUNT(DISTINCT o.order_id) AS orders
FROM {SILVER}.sales_orders o
WHERE date_format(o.order_date, 'yyyy-MM') = '{p_year_month}'
GROUP BY date_format(o.order_date, 'yyyy-MM'), o.channel, o.status
ORDER BY channel, status
""").show()

## 3. Pricing and warehouse decision

> Trainer TODO before delivery: capture a current official Databricks Pricing
> or Pricing Calculator screenshot and save it as
> `assets/images/databricks_pricing_YYYY-MM-DD.png`.

For Medi, keep this decision-level. Participants do not need to memorize every
price; they need to understand which choices change cost.

The practical pricing conversation has three parts:

1. **Unit price evidence** - official Databricks pricing or pricing calculator.
2. **Workload behavior** - how long the warehouse runs, how many users query it,
   and whether Power BI uses Import or DirectQuery.
3. **Model shape** - whether Power BI hits a narrow Gold aggregate or scans a
   wide detail table repeatedly.

| Warehouse type | Best use in this course | Watch out |
|---|---|---|
| Serverless | default for interactive SQL and BI demos | fastest start, but query volume still matters |
| Pro | controlled environments that need classic warehouse behavior | more operational choices |
| Classic | legacy or constrained setups | slower operational experience |

Core cost drivers:

- **DBU rate x runtime** - how long the warehouse is active.
- **Warehouse size** - larger warehouse costs more per active hour.
- **Auto-stop** - controls idle cost.
- **Concurrency** - many users and many Power BI visuals can create bursts.
- **Query shape** - bad SQL can cost more than the warehouse choice.

| Decision | Import mode tendency | DirectQuery tendency |
|---|---|---|
| Warehouse size | smaller is often enough | needs low latency |
| Auto-stop | aggressive auto-stop is fine | too aggressive can hurt user clicks |
| Query volume | scheduled refreshes | every visual and filter interaction |
| Best Gold source | aggregate table | narrow, selective Gold table/view |
| Cost risk | refresh window | user interaction fan-out |

### Optional: cost monitoring orientation

If the workspace has access to system tables, show the idea of cost
monitoring. Do not make this a required lab, because permissions vary by
workspace.

In [ ]:
# Optional orientation only. This may require account-level/system table access.
try:
    spark.sql("""
    SELECT
      cloud,
      sku_name,
      usage_unit,
      ROUND(SUM(usage_quantity), 2) AS usage_quantity
    FROM system.billing.usage
    WHERE usage_start_time >= current_date() - INTERVAL 30 DAYS
    GROUP BY cloud, sku_name, usage_unit
    ORDER BY usage_quantity DESC
    LIMIT 10
    """).show(truncate=False)
except Exception as e:
    print("[INFO] system.billing.usage is not available in this workspace or for this user.")
    print("Use this as a UI discussion point instead of a required lab.")
    print(type(e).__name__, str(e)[:240])

## 4. Unity Catalog and Catalog Explorer orientation

![Catalog Explorer lineage](../../assets/images/source_catalog_explorer_lineage.png)

For this course, Unity Catalog is not an admin topic. It is the way analysts
discover trusted objects and understand ownership.

For Power BI developers, Catalog Explorer answers the questions that usually
get lost in report handoff:

| Catalog Explorer signal | What it tells the BI developer |
|---|---|
| table comment | whether this object is meant for BI consumption |
| column comments | which fields are safe for slicers, labels and measures |
| owner | who approves metric definitions or investigates data issues |
| lineage | which upstream tables feed a dashboard number |
| permissions | whether the report can be refreshed by the intended identity |

`[UI DEMO]` Show in Catalog Explorer:

1. catalog -> schema -> table hierarchy,
2. comments and table metadata,
3. lineage for a Gold object,
4. where a Power BI developer would find the object name to connect to.

In [ ]:
print("Active catalog and schema")
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

print("Gold objects visible to this notebook")
spark.sql(f"SHOW TABLES IN {GOLD}").show(truncate=False)

## 5. Source data discovery

The first technical task is not to build Gold. It is to understand source
objects, grain and risk.

Discovery questions:

- What is the business entity?
- What is the grain?
- Which column is the stable key?
- Which columns are safe for filtering?
- Which values need business clarification?

In [ ]:
for table_name in ["customers", "products", "sales_orders", "order_lines"]:
    full_name = f"{SILVER}.{table_name}"
    rows = spark.sql(f"SELECT COUNT(*) AS n FROM {full_name}").first()["n"]
    print(f"{full_name}: {rows:,} rows")

In [ ]:
spark.sql(f"""
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS orders,
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(DISTINCT customer_id) AS distinct_customers
FROM {SILVER}.sales_orders
""").show(truncate=False)

spark.sql(f"""
SELECT status, COUNT(*) AS rows
FROM {SILVER}.sales_orders
GROUP BY status
ORDER BY rows DESC
""").show()

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS line_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_lines_per_order
FROM {SILVER}.order_lines
""").show()

## 6. Data map and grain discussion

![RetailHub source map](../../assets/images/retailhub_source_map.png)

Fill this table as a group:

| Source object | Business entity | Grain | Key columns | Main risks |
|---|---|---|---|---|
| `silver.customers` | customer | one row per customer | `customer_id` | missing segment/region |
| `silver.products` | product | one row per product | `product_id` | missing category/cost |
| `silver.sales_orders` | order header | one row per order | `order_id` | status semantics |
| `silver.order_lines` | order line | one row per order-product line | `line_id` | joins, quantity, price |

### Common mistakes to catch early

| Mistake | Symptom later in BI | Prevention |
|---|---|---|
| Treating order lines as orders | inflated order count | use `COUNT(DISTINCT order_id)` |
| Joining dimensions before checking grain | duplicated revenue | validate keys and row counts before/after join |
| Building dashboard on Silver directly | duplicated KPI logic in reports | build Gold contract first |
| Ignoring status semantics | revenue includes cancelled or returned orders | agree KPI filters in Gold |
| Querying every column for BI | slow visuals and larger scans | select only columns required by report |

## 7. Medallion Architecture

Bronze, Silver and Gold are not just quality labels. They are responsibilities.

| Layer | Main responsibility | Typical consumer | Should contain |
|---|---|---|---|
| Bronze | raw landing and traceability | data engineering | raw files/tables, ingestion metadata |
| Silver | cleaned and conformed entities | analytics engineering | reusable business entities |
| Gold | curated analytics contracts | BI, dashboards, Genie | stable facts, dimensions, KPI aggregates |

	Gold should not be a dumping ground for every possible column. It should be a
	clear contract for a defined analytical use case.
	

## 8. Where Medallion fits among architecture patterns

![Architecture patterns to Power BI Gold](../../assets/images/architecture_patterns_to_powerbi.png)

Power BI developers often hear multiple architecture terms in the same
conversation: Medallion, Kimball, Inmon, Data Vault and Data Mesh. They are not
all competing answers to the same question.

| Pattern | Main idea | What a Power BI developer should remember |
|---|---|---|
| Medallion | Bronze/Silver/Gold quality and consumption layers in the lakehouse | practical Databricks layering; Power BI usually consumes Gold |
| Kimball | dimensional model with facts and dimensions | best default mental model for BI reporting and slicers |
| Inmon | enterprise data warehouse with an integrated, normalized core | useful enterprise modelling idea; often upstream from BI marts |
| Data Vault | historized integration model: hubs, links, satellites, business vault | strong for audit/history and many sources; Power BI should not usually query Raw Vault directly |
| Data Mesh | operating model: domain ownership and data products | helps define who owns Gold objects, contracts and quality |

Key point: **Medallion is a lakehouse layering pattern. Kimball, Inmon and Data
Vault are modelling approaches. Data Mesh is an operating model.** In this
training, Gold is where these ideas become a usable BI contract.

### Decision guide for this training

| Situation | Good default |
|---|---|
| Building a Power BI report source | Gold Kimball-style fact/dim or aggregate |
| Integrating many operational systems with full audit history | Data Vault upstream, then publish BI-friendly Gold |
| Creating a central enterprise model | Inmon-style integrated core can inform Silver/Gold design |
| Assigning ownership and quality responsibility | Data Mesh data product thinking |
| Teaching Databricks Lakehouse flow | Medallion from Bronze to Silver to Gold |

For this course we keep the hands-on path simple: Medallion explains the layers,
Kimball shapes the BI-facing Gold model, and Data Mesh gives us ownership and
contract language.

## 9. Gold as a BI contract

![Gold business value](../../assets/images/gold_business_value.png)

A BI contract answers practical questions:

- What is the grain?
- Which filters define each KPI?
- Which dimensions are supported?
- Which rows are excluded?
- Who owns the definition?
- What is the refresh expectation?
- Which object should Power BI query?

If this is not written in Databricks, the logic usually leaks into many Power
BI reports and becomes impossible to govern.

## 10. Kimball-style Gold model

![Kimball Gold model](../../assets/images/kimball_gold_model.png)

We use a simple star-schema pattern:

- dimensions describe filtering and grouping,
- facts store measurable events,
- aggregates support fast dashboard pages,
- views can support parameterized or incremental access patterns.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_date
COMMENT 'Date dimension for RetailHub BI reports.'
AS
SELECT DISTINCT
  order_date AS date_key,
  year(order_date) AS year,
  quarter(order_date) AS quarter,
  month(order_date) AS month,
  date_format(order_date, 'yyyy-MM') AS year_month,
  dayofweek(order_date) AS day_of_week
FROM {SILVER}.sales_orders
WHERE order_date IS NOT NULL
""")

spark.sql(f"SELECT * FROM {GOLD}.dim_date ORDER BY date_key LIMIT 10").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_product
COMMENT 'Product dimension for BI slicers and category analysis.'
AS
SELECT
  product_id,
  product_name,
  category,
  subcategory,
  unit_cost
FROM {SILVER}.products
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_customer
COMMENT 'Customer dimension for BI slicers and regional analysis.'
AS
SELECT
  customer_id,
  customer_name,
  segment,
  region AS customer_region
FROM {SILVER}.customers
""")

print("[OK] Dimensions refreshed")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales
COMMENT 'Gold sales fact at order-line grain. One row per sales order line.'
AS
SELECT
  l.line_id,
  l.order_id,
  o.customer_id,
  l.product_id,
  o.order_date,
  o.status,
  o.channel,
  l.quantity,
  l.unit_price,
  p.unit_cost,
  CASE WHEN o.status = 'Completed' THEN true ELSE false END AS is_completed,
  l.quantity * l.unit_price AS line_revenue,
  l.quantity * (l.unit_price - p.unit_cost) AS line_margin
FROM {SILVER}.order_lines l
JOIN {SILVER}.sales_orders o ON l.order_id = o.order_id
LEFT JOIN {SILVER}.products p ON l.product_id = p.product_id
""")

spark.sql(f"SELECT * FROM {GOLD}.fact_sales LIMIT 10").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard
COMMENT 'BI-ready detail table joining sales fact with conformed dimensions.'
AS
SELECT
  f.line_id,
  f.order_id,
  f.order_date,
  d.year,
  d.quarter,
  d.month,
  d.year_month,
  f.status,
  f.channel,
  f.is_completed,
  f.quantity,
  f.unit_price,
  f.unit_cost,
  f.line_revenue,
  f.line_margin,
  c.customer_id,
  c.customer_name,
  c.segment,
  c.customer_region,
  p.product_id,
  p.product_name,
  p.category,
  p.subcategory
FROM {GOLD}.fact_sales f
LEFT JOIN {GOLD}.dim_date d ON f.order_date = d.date_key
LEFT JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
LEFT JOIN {GOLD}.dim_product p ON f.product_id = p.product_id
""")

spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.fact_sales_dashboard").show()

## 11. Reconciliation and quality gates

![Gold quality gate](../../assets/images/gold_quality_gate.png)

Gold tables should be checked before they become report sources. The checks
below are intentionally simple and readable for analysts.

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer,
  SUM(CASE WHEN unit_price IS NULL THEN 1 ELSE 0 END) AS missing_price
FROM {GOLD}.fact_sales_dashboard
""").show()

In [ ]:
spark.sql(f"""
WITH fact_total AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales
),
dashboard_total AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard
)
SELECT
  f.revenue AS fact_revenue,
  d.revenue AS dashboard_revenue,
  ROUND(f.revenue - d.revenue, 2) AS diff
FROM fact_total f CROSS JOIN dashboard_total d
""").show()

In [ ]:
quality = spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_order_date,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id
FROM {GOLD}.fact_sales_dashboard
""").first()

assert quality["rows"] > 0, "Gold dashboard table is empty"
assert quality["rows"] == quality["distinct_lines"], "Gold dashboard grain is not unique at line_id"
assert quality["missing_order_date"] == 0, "Gold dashboard has missing order_date"

print("[OK] Gold dashboard table is non-empty, line-grain unique and date-complete")
print(dict(quality.asDict()))

## 12. Optional bonus: SCD in analyst context

Slowly changing dimensions matter when a customer or product attribute changes
and historical reporting must preserve the old value. This is not a data
engineering detail only; it changes what Power BI users see in historical
trends and slicers.

Example: a customer moved from `SMB` to `Enterprise` in May. Should January
revenue move to Enterprise too, or should January remain SMB? That is a model
decision, not a visual formatting decision.

| Question | Type 1 tendency | Type 2 tendency |
|---|---|---|
| Does history matter? | no | yes |
| Example | correct a misspelled segment | customer moved between segments |
| BI impact | old reports change | old reports remain historically true |
| Complexity | lower | higher |

Keep this as a discussion or 15-minute extension. It should not distract from
the main goal: a stable Gold layer for BI.

## Summary and artefacts

Created or refreshed:

- `gold.dim_date`
- `gold.dim_product`
- `gold.dim_customer`
- `gold.fact_sales`
- `gold.fact_sales_dashboard`

Prepared for the next notebook:

- SQL Editor workflow,
- warehouse cost decision,
- Catalog Explorer / Unity Catalog orientation,
- source grain understanding,
- Medallion compared with Kimball, Inmon, Data Vault and Data Mesh,
- Gold BI contract discussion,
- data-quality risks,
- first reconciliation pattern.